In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


# ---------------------------------------------------------
# 1. FIND THE MAIN PROJECT FOLDER
# ---------------------------------------------------------

ROOT = Path.cwd().parents[1]


# ---------------------------------------------------------
# 2. DEFINE IMPORTANT DATA FOLDERS
# ---------------------------------------------------------

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"


# ---------------------------------------------------------
# 3. MAKE SURE THE PROCESSED FOLDER EXISTS
# ---------------------------------------------------------

PROCESSED.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# 4. PRINT PATHS SO WE CAN CHECK THEM
# ---------------------------------------------------------

print("Project root:")
print(ROOT)

print("\nRaw data folder:")
print(RAW)

print("\nProcessed data folder:")
print(PROCESSED)

Project root:
/Users/ansh/Desktop/ControlSight

Raw data folder:
/Users/ansh/Desktop/ControlSight/data/raw

Processed data folder:
/Users/ansh/Desktop/ControlSight/data/processed


In [2]:
projects_raw = pd.read_csv(
    RAW / "projects.csv"
)

vendors_raw = pd.read_csv(
    RAW / "vendors.csv"
)

packages_raw = pd.read_csv(
    RAW / "cost_packages.csv"
)

costs_raw = pd.read_csv(
    RAW / "cost_monthly.csv"
)

schedule_raw = pd.read_csv(
    RAW / "schedule_activities.csv"
)

procurement_raw = pd.read_csv(
    RAW / "procurement_packages.csv"
)

progress_raw = pd.read_csv(
    RAW / "progress_updates.csv"
)

print("All raw datasets loaded successfully.")

All raw datasets loaded successfully.


In [3]:
datasets = {
    "projects": projects_raw,
    "vendors": vendors_raw,
    "cost_packages": packages_raw,
    "cost_monthly": costs_raw,
    "schedule_activities": schedule_raw,
    "procurement_packages": procurement_raw,
    "progress_updates": progress_raw,
}

for name, df in datasets.items():
    print(
        f"{name:25s}"
        f"{len(df):5d} rows x "
        f"{len(df.columns):2d} columns"
    )

projects                     8 rows x 10 columns
vendors                     30 rows x  4 columns
cost_packages               96 rows x  5 columns
cost_monthly              1824 rows x 11 columns
schedule_activities        320 rows x 13 columns
procurement_packages       119 rows x 13 columns
progress_updates           152 rows x  6 columns


In [4]:
for name, df in datasets.items():

    print("=" * 80)
    print(name.upper())
    print("=" * 80)

    print("\nShape:")
    print(df.shape)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicate rows:")
    print(df.duplicated().sum())

    print("\nData types:")
    print(df.dtypes)

    print("\n")

PROJECTS

Shape:
(8, 10)

Missing values:
project_code        0
project_name        0
project_type        0
region              0
start_date          0
planned_end_date    0
approved_budget     0
project_manager     0
status              0
currency            0
dtype: int64

Duplicate rows:
0

Data types:
project_code        object
project_name        object
project_type        object
region              object
start_date          object
planned_end_date    object
approved_budget      int64
project_manager     object
status              object
currency            object
dtype: object


VENDORS

Shape:
(30, 4)

Missing values:
vendor_id          0
vendor_name        0
vendor_category    0
country            0
dtype: int64

Duplicate rows:
0

Data types:
vendor_id          object
vendor_name        object
vendor_category    object
country            object
dtype: object


COST_PACKAGES

Shape:
(96, 5)

Missing values:
package_code      0
project_code      0
package_name      0
cost_categ

In [5]:
audit_summary = []

for name, df in datasets.items():

    audit_summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    })

audit_summary = pd.DataFrame(audit_summary)

audit_summary

,dataset,rows,columns,missing_values,duplicate_rows
0,projects,8,10,0,0
1,vendors,30,4,0,0
2,cost_packages,96,5,0,0
3,cost_monthly,1824,11,1,0
4,schedule_activities,320,13,15,0
5,procurement_packages,119,13,3,1
6,progress_updates,152,6,0,0


In [6]:
quality_issues = []

def add_issue(dataset, row_index, issue_type, column, value, message):
    quality_issues.append({
        "dataset": dataset,
        "row_index": row_index,
        "issue_type": issue_type,
        "column": column,
        "value": value,
        "message": message,
    })

In [7]:
# ---------------------------------------------------------
# COST DATA QUALITY CHECKS
# ---------------------------------------------------------

valid_project_codes = set(
    projects_raw["project_code"]
)

valid_package_codes = set(
    packages_raw["package_code"]
)


for idx, row in costs_raw.iterrows():

    # 1. Project code must exist
    if row["project_code"] not in valid_project_codes:
        add_issue(
            dataset="cost_monthly",
            row_index=idx,
            issue_type="invalid_reference",
            column="project_code",
            value=row["project_code"],
            message="Project code does not exist in projects.csv",
        )

    # 2. Package code is required
    if pd.isna(row["package_code"]):
        add_issue(
            dataset="cost_monthly",
            row_index=idx,
            issue_type="missing_value",
            column="package_code",
            value=row["package_code"],
            message="Package code is missing",
        )

    # 3. Package code must exist
    elif row["package_code"] not in valid_package_codes:
        add_issue(
            dataset="cost_monthly",
            row_index=idx,
            issue_type="invalid_reference",
            column="package_code",
            value=row["package_code"],
            message="Package code does not exist in cost_packages.csv",
        )

    # 4. Actual cost should not be negative
    if row["actual_cost"] < 0:
        add_issue(
            dataset="cost_monthly",
            row_index=idx,
            issue_type="invalid_value",
            column="actual_cost",
            value=row["actual_cost"],
            message="Actual cost cannot be negative",
        )


print("Cost quality checks completed.")

Cost quality checks completed.


In [8]:
quality_issues_df = pd.DataFrame(
    quality_issues
)

quality_issues_df

,dataset,row_index,issue_type,column,value,message
0,cost_monthly,17,invalid_reference,project_code,PRJ-999,Project code does not exist in projects.csv
1,cost_monthly,80,invalid_value,actual_cost,-25000.0,Actual cost cannot be negative
2,cost_monthly,180,missing_value,package_code,NaN,Package code is missing


In [9]:
# ---------------------------------------------------------
# PROGRESS DATA QUALITY CHECKS
# ---------------------------------------------------------

for idx, row in progress_raw.iterrows():

    # 1. Project code must exist
    if row["project_code"] not in valid_project_codes:
        add_issue(
            dataset="progress_updates",
            row_index=idx,
            issue_type="invalid_reference",
            column="project_code",
            value=row["project_code"],
            message="Project code does not exist in projects.csv",
        )

    # 2. Actual progress must be between 0 and 100
    if (
        row["actual_progress_pct"] < 0
        or row["actual_progress_pct"] > 100
    ):
        add_issue(
            dataset="progress_updates",
            row_index=idx,
            issue_type="invalid_value",
            column="actual_progress_pct",
            value=row["actual_progress_pct"],
            message="Actual progress must be between 0 and 100",
        )

    # 3. Planned progress must be between 0 and 100
    if (
        row["planned_progress_pct"] < 0
        or row["planned_progress_pct"] > 100
    ):
        add_issue(
            dataset="progress_updates",
            row_index=idx,
            issue_type="invalid_value",
            column="planned_progress_pct",
            value=row["planned_progress_pct"],
            message="Planned progress must be between 0 and 100",
        )

    # 4. Check whether report_month can be interpreted as a date
    try:
        pd.to_datetime(
            row["report_month"],
            format="%Y-%m-%d",
            errors="raise",
        )

    except (ValueError, TypeError):

        add_issue(
            dataset="progress_updates",
            row_index=idx,
            issue_type="invalid_format",
            column="report_month",
            value=row["report_month"],
            message="Report month is not in YYYY-MM-DD format",
        )


print("Progress quality checks completed.")

Progress quality checks completed.


In [10]:
quality_issues_df = pd.DataFrame(
    quality_issues
)

quality_issues_df

,dataset,row_index,issue_type,column,value,message
0,cost_monthly,17,invalid_reference,project_code,PRJ-999,Project code does not exist in projects.csv
1,cost_monthly,80,invalid_value,actual_cost,-25000.0,Actual cost cannot be negative
2,cost_monthly,180,missing_value,package_code,NaN,Package code is missing
3,progress_updates,9,invalid_format,report_month,31/12/2025,Report month is not in YYYY-MM-DD format
4,progress_updates,25,invalid_value,actual_progress_pct,126.0,Actual progress must be between 0 and 100


In [11]:
# ---------------------------------------------------------
# SCHEDULE DATA QUALITY CHECKS
# ---------------------------------------------------------

for idx, row in schedule_raw.iterrows():

    # 1. Project code must exist
    if row["project_code"] not in valid_project_codes:
        add_issue(
            dataset="schedule_activities",
            row_index=idx,
            issue_type="invalid_reference",
            column="project_code",
            value=row["project_code"],
            message="Project code does not exist in projects.csv",
        )

    # 2. Percent complete should not be missing
    if pd.isna(row["percent_complete"]):
        add_issue(
            dataset="schedule_activities",
            row_index=idx,
            issue_type="missing_value",
            column="percent_complete",
            value=row["percent_complete"],
            message="Percent complete is missing",
        )

    # 3. Percent complete must be between 0 and 100
    elif (
        row["percent_complete"] < 0
        or row["percent_complete"] > 100
    ):
        add_issue(
            dataset="schedule_activities",
            row_index=idx,
            issue_type="invalid_value",
            column="percent_complete",
            value=row["percent_complete"],
            message="Percent complete must be between 0 and 100",
        )

    # 4. Actual finish must not be before actual start
    if (
        pd.notna(row["actual_start"])
        and pd.notna(row["actual_finish"])
    ):

        actual_start = pd.to_datetime(
            row["actual_start"],
            errors="coerce",
        )

        actual_finish = pd.to_datetime(
            row["actual_finish"],
            errors="coerce",
        )

        if (
            pd.notna(actual_start)
            and pd.notna(actual_finish)
            and actual_finish < actual_start
        ):
            add_issue(
                dataset="schedule_activities",
                row_index=idx,
                issue_type="logical_error",
                column="actual_finish",
                value=row["actual_finish"],
                message="Actual finish date occurs before actual start date",
            )


print("Schedule quality checks completed.")

Schedule quality checks completed.


In [12]:
quality_issues_df = pd.DataFrame(
    quality_issues
)

quality_issues_df

,dataset,row_index,issue_type,column,value,message
0,cost_monthly,17,invalid_reference,project_code,PRJ-999,Project code does not exist in projects.csv
1,cost_monthly,80,invalid_value,actual_cost,-25000.0,Actual cost cannot be negative
2,cost_monthly,180,missing_value,package_code,NaN,Package code is missing
3,progress_updates,9,invalid_format,report_month,31/12/2025,Report month is not in YYYY-MM-DD format
4,progress_updates,25,invalid_value,actual_progress_pct,126.0,Actual progress must be between 0 and 100
5,schedule_activities,30,logical_error,actual_finish,2026-06-10,Actual finish date occurs before actual start ...
6,schedule_activities,65,missing_value,percent_complete,NaN,Percent complete is missing


In [13]:
# ---------------------------------------------------------
# PROCUREMENT DATA QUALITY CHECKS
# ---------------------------------------------------------

valid_vendor_ids = set(
    vendors_raw["vendor_id"]
)

for idx, row in procurement_raw.iterrows():

    # 1. Project code must exist
    if row["project_code"] not in valid_project_codes:
        add_issue(
            dataset="procurement_packages",
            row_index=idx,
            issue_type="invalid_reference",
            column="project_code",
            value=row["project_code"],
            message="Project code does not exist in projects.csv",
        )

    # 2. Package code must exist
    if row["package_code"] not in valid_package_codes:
        add_issue(
            dataset="procurement_packages",
            row_index=idx,
            issue_type="invalid_reference",
            column="package_code",
            value=row["package_code"],
            message="Package code does not exist in cost_packages.csv",
        )

    # 3. Vendor ID is required
    if pd.isna(row["vendor_id"]):
        add_issue(
            dataset="procurement_packages",
            row_index=idx,
            issue_type="missing_value",
            column="vendor_id",
            value=row["vendor_id"],
            message="Vendor ID is missing",
        )

    # 4. Vendor ID must exist
    elif row["vendor_id"] not in valid_vendor_ids:
        add_issue(
            dataset="procurement_packages",
            row_index=idx,
            issue_type="invalid_reference",
            column="vendor_id",
            value=row["vendor_id"],
            message="Vendor ID does not exist in vendors.csv",
        )

In [14]:
duplicate_procurement = procurement_raw[
    procurement_raw.duplicated(
        subset=["procurement_id"],
        keep=False,
    )
]

for idx, row in duplicate_procurement.iterrows():

    add_issue(
        dataset="procurement_packages",
        row_index=idx,
        issue_type="duplicate_record",
        column="procurement_id",
        value=row["procurement_id"],
        message="Duplicate procurement ID detected",
    )

print("Procurement quality checks completed.")

Procurement quality checks completed.


In [15]:
quality_issues_df = pd.DataFrame(
    quality_issues
)

quality_issues_df

,dataset,row_index,issue_type,column,value,message
0,cost_monthly,17,invalid_reference,project_code,PRJ-999,Project code does not exist in projects.csv
1,cost_monthly,80,invalid_value,actual_cost,-25000.0,Actual cost cannot be negative
2,cost_monthly,180,missing_value,package_code,NaN,Package code is missing
3,progress_updates,9,invalid_format,report_month,31/12/2025,Report month is not in YYYY-MM-DD format
4,progress_updates,25,invalid_value,actual_progress_pct,126.0,Actual progress must be between 0 and 100
5,schedule_activities,30,logical_error,actual_finish,2026-06-10,Actual finish date occurs before actual start ...
6,schedule_activities,65,missing_value,percent_complete,NaN,Percent complete is missing
7,procurement_packages,4,missing_value,vendor_id,NaN,Vendor ID is missing
8,procurement_packages,56,missing_value,vendor_id,NaN,Vendor ID is missing
9,procurement_packages,89,missing_value,vendor_id,NaN,Vendor ID is missing


In [ ]:
# ---------------------------------------------------------
# COST ANOMALY CHECK
# ---------------------------------------------------------

for idx, row in costs_raw.iterrows():

    # Avoid division by zero
    if row["planned_cost"] > 0:

        cost_ratio = (
            row["actual_cost"]
            / row["planned_cost"]
        )

        # Flag unusually high actual cost
        if cost_ratio > 3:

           add_issue(
                dataset="cost_monthly",
                row_index=idx,
                issue_type="anomaly",
                column="actual_cost",
                value=row["actual_cost"],
                message=(
                    "Actual cost is more than 3 times "
                    "the planned cost"
                ),
            )


print("Cost anomaly checks completed.")

Cost anomaly checks completed.


In [17]:
quality_issues_df = pd.DataFrame(
    quality_issues
)

quality_issues_df

,dataset,row_index,issue_type,column,value,message
0,cost_monthly,17,invalid_reference,project_code,PRJ-999,Project code does not exist in projects.csv
1,cost_monthly,80,invalid_value,actual_cost,-25000.0,Actual cost cannot be negative
2,cost_monthly,180,missing_value,package_code,NaN,Package code is missing
3,progress_updates,9,invalid_format,report_month,31/12/2025,Report month is not in YYYY-MM-DD format
4,progress_updates,25,invalid_value,actual_progress_pct,126.0,Actual progress must be between 0 and 100
5,schedule_activities,30,logical_error,actual_finish,2026-06-10,Actual finish date occurs before actual start ...
6,schedule_activities,65,missing_value,percent_complete,NaN,Percent complete is missing
7,procurement_packages,4,missing_value,vendor_id,NaN,Vendor ID is missing
8,procurement_packages,56,missing_value,vendor_id,NaN,Vendor ID is missing
9,procurement_packages,89,missing_value,vendor_id,NaN,Vendor ID is missing


In [18]:
# ---------------------------------------------------------
# DATA QUALITY SUMMARY
# ---------------------------------------------------------

quality_summary = (
    quality_issues_df
    .groupby(
        [
            "dataset",
            "issue_type",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="issue_count"
    )
    .sort_values(
        [
            "dataset",
            "issue_count",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

quality_summary

,dataset,issue_type,issue_count
0,cost_monthly,anomaly,1
1,cost_monthly,invalid_reference,1
2,cost_monthly,invalid_value,1
3,cost_monthly,missing_value,1
5,procurement_packages,missing_value,3
4,procurement_packages,duplicate_record,2
6,progress_updates,invalid_format,1
7,progress_updates,invalid_value,1
8,schedule_activities,logical_error,1
9,schedule_activities,missing_value,1


In [19]:
print(
    "Total data-quality issues:",
    len(quality_issues_df)
)

print()

print(
    "Affected datasets:",
    quality_issues_df["dataset"].nunique()
)

Total data-quality issues: 13

Affected datasets: 4


In [20]:
# ---------------------------------------------------------
# CREATE CLEAN COPIES
# ---------------------------------------------------------

projects_clean = projects_raw.copy()
vendors_clean = vendors_raw.copy()
packages_clean = packages_raw.copy()
costs_clean = costs_raw.copy()
schedule_clean = schedule_raw.copy()
procurement_clean = procurement_raw.copy()
progress_clean = progress_raw.copy()

print("Clean working copies created.")

Clean working copies created.


In [51]:
# ---------------------------------------------------------
# STANDARDISE DATE COLUMNS
# ---------------------------------------------------------

projects_clean["start_date"] = pd.to_datetime(
    projects_clean["start_date"],
    errors="coerce"
)

projects_clean["planned_end_date"] = pd.to_datetime(
    projects_clean["planned_end_date"],
    errors="coerce"
)

costs_clean["report_month"] = pd.to_datetime(
    costs_clean["report_month"],
    errors="coerce",
    dayfirst=True
)

progress_clean["report_month"] = pd.to_datetime(
    progress_clean["report_month"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

schedule_clean["planned_start"] = pd.to_datetime(
    schedule_clean["planned_start"],
    errors="coerce"
)

schedule_clean["planned_finish"] = pd.to_datetime(
    schedule_clean["planned_finish"],
    errors="coerce"
)

schedule_clean["actual_start"] = pd.to_datetime(
    schedule_clean["actual_start"],
    errors="coerce"
)

schedule_clean["actual_finish"] = pd.to_datetime(
    schedule_clean["actual_finish"],
    errors="coerce"
)

procurement_clean["rfq_date"] = pd.to_datetime(
    procurement_clean["rfq_date"],
    errors="coerce"
)

procurement_clean["po_date"] = pd.to_datetime(
    procurement_clean["po_date"],
    errors="coerce"
)

procurement_clean["planned_delivery_date"] = pd.to_datetime(
    procurement_clean["planned_delivery_date"],
    errors="coerce"
)

procurement_clean["actual_delivery_date"] = pd.to_datetime(
    procurement_clean["actual_delivery_date"],
    errors="coerce"
)

print("Date columns standardised.")

Date columns standardised.


In [52]:
progress_clean.loc[9]

project_code                        PRJ-001
report_month            2025-12-31 00:00:00
planned_progress_pct                  41.67
actual_progress_pct                   42.91
reported_by                       Site Team
source_system                     P6_EXPORT
Name: 9, dtype: object

In [50]:
progress_clean = progress_raw.copy()

print(progress_clean.loc[9, "report_month"])

31/12/2025


In [53]:
print(
    "Missing progress dates:",
    progress_clean["report_month"].isna().sum()
)

Missing progress dates: 0


In [54]:
quarantine_progress_indices = set()

for idx, row in progress_clean.iterrows():

    if (
        row["actual_progress_pct"] < 0
        or row["actual_progress_pct"] > 100
    ):
        quarantine_progress_indices.add(idx)

    if row["project_code"] not in valid_project_codes:
        quarantine_progress_indices.add(idx)


progress_quarantine = progress_clean.loc[
    sorted(quarantine_progress_indices)
].copy()

progress_clean = progress_clean.drop(
    index=quarantine_progress_indices
).copy()

print(
    "Clean progress rows:",
    len(progress_clean)
)

print(
    "Quarantined progress rows:",
    len(progress_quarantine)
)

Clean progress rows: 151
Quarantined progress rows: 1


In [55]:
progress_clean["progress_variance_pct"] = (
    progress_clean["actual_progress_pct"]
    - progress_clean["planned_progress_pct"]
)

progress_clean["behind_schedule_flag"] = (
    progress_clean["progress_variance_pct"] < -5
)

In [56]:
progress_clean.to_csv(
    PROCESSED / "progress_updates_clean.csv",
    index=False
)

progress_quarantine.to_csv(
    PROCESSED / "progress_updates_quarantine.csv",
    index=False
)

print("Corrected progress files saved.")

Corrected progress files saved.


In [57]:
test_progress = pd.read_csv(
    PROCESSED / "progress_updates_clean.csv"
)

print(
    "Missing dates in saved file:",
    test_progress["report_month"].isna().sum()
)

test_progress.head(12)

Missing dates in saved file: 0


,project_code,report_month,planned_progress_pct,actual_progress_pct,reported_by,source_system,progress_variance_pct,behind_schedule_flag
0,PRJ-001,2025-01-31,4.17,4.17,Site Team,P6_EXPORT,0.00,False
1,PRJ-001,2025-02-28,8.33,11.86,Project Controls,P6_EXPORT,3.53,False
2,PRJ-001,2025-03-31,12.50,12.97,Project Controls,P6_EXPORT,0.47,False
3,PRJ-001,2025-04-30,16.67,15.94,Site Team,P6_EXPORT,-0.73,False
4,PRJ-001,2025-05-31,20.83,20.36,Site Team,P6_EXPORT,-0.47,False
5,PRJ-001,2025-06-30,25.00,22.75,Project Controls,P6_EXPORT,-2.25,False
6,PRJ-001,2025-07-31,29.17,30.30,Project Controls,P6_EXPORT,1.13,False
7,PRJ-001,2025-08-31,33.33,33.79,PMO,P6_EXPORT,0.46,False
8,PRJ-001,2025-09-30,37.50,35.55,Site Team,P6_EXPORT,-1.95,False
9,PRJ-001,2025-12-31,41.67,42.91,Site Team,P6_EXPORT,1.24,False


In [22]:
# ---------------------------------------------------------
# CREATE QUARANTINE LISTS
# ---------------------------------------------------------

quarantine_cost_indices = set()
quarantine_progress_indices = set()
quarantine_schedule_indices = set()
quarantine_procurement_indices = set()

In [23]:
# ---------------------------------------------------------
# IDENTIFY COST RECORDS TO QUARANTINE
# ---------------------------------------------------------

for idx, row in costs_clean.iterrows():

    # Invalid project reference
    if row["project_code"] not in valid_project_codes:
        quarantine_cost_indices.add(idx)

    # Missing package
    if pd.isna(row["package_code"]):
        quarantine_cost_indices.add(idx)

    # Negative cost
    if row["actual_cost"] < 0:
        quarantine_cost_indices.add(idx)


print(
    "Cost records to quarantine:",
    len(quarantine_cost_indices)
)

Cost records to quarantine: 3


In [24]:
# ---------------------------------------------------------
# IDENTIFY PROGRESS RECORDS TO QUARANTINE
# ---------------------------------------------------------

for idx, row in progress_clean.iterrows():

    # Invalid percentage
    if (
        row["actual_progress_pct"] < 0
        or row["actual_progress_pct"] > 100
    ):
        quarantine_progress_indices.add(idx)

    # Invalid project
    if row["project_code"] not in valid_project_codes:
        quarantine_progress_indices.add(idx)


print(
    "Progress records to quarantine:",
    len(quarantine_progress_indices)
)

Progress records to quarantine: 1


In [25]:
# ---------------------------------------------------------
# IDENTIFY SCHEDULE RECORDS TO QUARANTINE
# ---------------------------------------------------------

for idx, row in schedule_clean.iterrows():

    # Missing completion percentage
    if pd.isna(row["percent_complete"]):
        quarantine_schedule_indices.add(idx)

    # Impossible date relationship
    if (
        pd.notna(row["actual_start"])
        and pd.notna(row["actual_finish"])
        and row["actual_finish"] < row["actual_start"]
    ):
        quarantine_schedule_indices.add(idx)


print(
    "Schedule records to quarantine:",
    len(quarantine_schedule_indices)
)

Schedule records to quarantine: 2


In [26]:
# ---------------------------------------------------------
# IDENTIFY PROCUREMENT RECORDS TO QUARANTINE
# ---------------------------------------------------------

for idx, row in procurement_clean.iterrows():

    # Missing vendor
    if pd.isna(row["vendor_id"]):
        quarantine_procurement_indices.add(idx)

    # Invalid vendor
    elif row["vendor_id"] not in valid_vendor_ids:
        quarantine_procurement_indices.add(idx)

    # Invalid project
    if row["project_code"] not in valid_project_codes:
        quarantine_procurement_indices.add(idx)

    # Invalid package
    if row["package_code"] not in valid_package_codes:
        quarantine_procurement_indices.add(idx)


print(
    "Procurement records to quarantine:",
    len(quarantine_procurement_indices)
)

Procurement records to quarantine: 3


In [27]:
# ---------------------------------------------------------
# SPLIT CLEAN AND QUARANTINED RECORDS
# ---------------------------------------------------------

# COSTS
costs_quarantine = costs_clean.loc[
    sorted(quarantine_cost_indices)
].copy()

costs_clean = costs_clean.drop(
    index=quarantine_cost_indices
).copy()


# PROGRESS
progress_quarantine = progress_clean.loc[
    sorted(quarantine_progress_indices)
].copy()

progress_clean = progress_clean.drop(
    index=quarantine_progress_indices
).copy()


# SCHEDULE
schedule_quarantine = schedule_clean.loc[
    sorted(quarantine_schedule_indices)
].copy()

schedule_clean = schedule_clean.drop(
    index=quarantine_schedule_indices
).copy()


# PROCUREMENT
procurement_quarantine = procurement_clean.loc[
    sorted(quarantine_procurement_indices)
].copy()

procurement_clean = procurement_clean.drop(
    index=quarantine_procurement_indices
).copy()


print("Clean and quarantined datasets created.")

Clean and quarantined datasets created.


In [28]:
print("COSTS")
print("Clean:", len(costs_clean))
print("Quarantined:", len(costs_quarantine))

print("\nPROGRESS")
print("Clean:", len(progress_clean))
print("Quarantined:", len(progress_quarantine))

print("\nSCHEDULE")
print("Clean:", len(schedule_clean))
print("Quarantined:", len(schedule_quarantine))

print("\nPROCUREMENT")
print("Clean:", len(procurement_clean))
print("Quarantined:", len(procurement_quarantine))

COSTS
Clean: 1821
Quarantined: 3

PROGRESS
Clean: 151
Quarantined: 1

SCHEDULE
Clean: 318
Quarantined: 2

PROCUREMENT
Clean: 116
Quarantined: 3


In [29]:
# ---------------------------------------------------------
# REMOVE DUPLICATE PROCUREMENT RECORDS
# ---------------------------------------------------------

before_duplicates = len(procurement_clean)

procurement_clean = procurement_clean.drop_duplicates(
    subset=["procurement_id"],
    keep="first"
).copy()

after_duplicates = len(procurement_clean)

duplicates_removed = (
    before_duplicates - after_duplicates
)

print(
    "Duplicate procurement records removed:",
    duplicates_removed
)

Duplicate procurement records removed: 1


In [30]:
# ---------------------------------------------------------
# DERIVED COST METRICS
# ---------------------------------------------------------

costs_clean["cost_variance"] = (
    costs_clean["planned_cost"]
    - costs_clean["actual_cost"]
)

costs_clean["cost_variance_pct"] = np.where(
    costs_clean["planned_cost"] != 0,
    (
        costs_clean["cost_variance"]
        / costs_clean["planned_cost"]
    ) * 100,
    np.nan
)

costs_clean["forecast_variance"] = (
    costs_clean["forecast_at_completion"]
    - costs_clean["planned_cost"]
)

costs_clean["cost_anomaly_flag"] = (
    costs_clean["actual_cost"]
    > (costs_clean["planned_cost"] * 3)
)

costs_clean.head()

,cost_record_id,project_code,package_code,cost_category,report_month,planned_cost,actual_cost,committed_cost,forecast_at_completion,currency,source_system,cost_variance,cost_variance_pct,forecast_variance,cost_anomaly_flag
0,C-000001,PRJ-001,PRJ-001-PKG-01,Civil,2025-01-31,200572.03,191419.61,206308.06,6745303.66,GBP,ERP_COST,9152.42,4.563159,6544731.63,False
1,C-000002,PRJ-001,PRJ-001-PKG-02,Structural,2025-01-31,85691.16,85236.76,96286.32,2788355.37,GBP,ERP_COST,454.40,0.530276,2702664.21,False
2,C-000003,PRJ-001,PRJ-001-PKG-03,Mechanical,2025-01-31,161080.60,167524.71,167524.71,5433249.95,GBP,ERP_COST,-6444.11,-4.000550,5272169.35,False
3,C-000004,PRJ-001,PRJ-001-PKG-04,Electrical,2025-01-31,129328.27,126722.65,147595.69,4204065.36,GBP,ERP_COST,2605.62,2.014734,4074737.09,False
4,C-000005,PRJ-001,PRJ-001-PKG-05,HVAC,2025-01-31,52316.67,47323.64,60821.26,1775476.36,GBP,ERP_COST,4993.03,9.543860,1723159.69,False


In [31]:
# ---------------------------------------------------------
# DERIVED PROGRESS METRICS
# ---------------------------------------------------------

progress_clean["progress_variance_pct"] = (
    progress_clean["actual_progress_pct"]
    - progress_clean["planned_progress_pct"]
)

progress_clean["behind_schedule_flag"] = (
    progress_clean["progress_variance_pct"] < -5
)

progress_clean.head()

,project_code,report_month,planned_progress_pct,actual_progress_pct,reported_by,source_system,progress_variance_pct,behind_schedule_flag
0,PRJ-001,2025-01-31,4.17,4.17,Site Team,P6_EXPORT,0.00,False
1,PRJ-001,2025-02-28,8.33,11.86,Project Controls,P6_EXPORT,3.53,False
2,PRJ-001,2025-03-31,12.50,12.97,Project Controls,P6_EXPORT,0.47,False
3,PRJ-001,2025-04-30,16.67,15.94,Site Team,P6_EXPORT,-0.73,False
4,PRJ-001,2025-05-31,20.83,20.36,Site Team,P6_EXPORT,-0.47,False


In [32]:
# ---------------------------------------------------------
# DERIVED SCHEDULE METRICS
# ---------------------------------------------------------

schedule_clean["start_delay_days"] = (
    schedule_clean["actual_start"]
    - schedule_clean["planned_start"]
).dt.days

schedule_clean["finish_delay_days"] = (
    schedule_clean["actual_finish"]
    - schedule_clean["planned_finish"]
).dt.days

schedule_clean["delayed_activity_flag"] = (
    schedule_clean["finish_delay_days"] > 0
)

schedule_clean.head()

,activity_id,project_code,wbs_code,activity_name,discipline,planned_start,planned_finish,actual_start,actual_finish,percent_complete,status,is_critical,source_system,start_delay_days,finish_delay_days,delayed_activity_flag
0,PRJ-001-ACT-001,PRJ-001,WBS-01,Design approval,Civil,2025-01-03,2025-02-06,2025-01-04,2025-02-14,100.0,Complete,False,P6_EXPORT,1.0,8.0,True
1,PRJ-001-ACT-002,PRJ-001,WBS-01,Site mobilisation,Structural,2025-01-16,2025-02-19,2025-01-19,2025-02-22,100.0,Complete,False,P6_EXPORT,3.0,3.0,True
2,PRJ-001-ACT-003,PRJ-001,WBS-01,Groundworks,Mechanical,2025-01-31,2025-03-13,2025-02-07,2025-03-17,100.0,Complete,False,P6_EXPORT,7.0,4.0,True
3,PRJ-001-ACT-004,PRJ-001,WBS-01,Structural frame,Electrical,2025-02-10,2025-03-13,2025-02-10,2025-03-13,100.0,Complete,False,P6_EXPORT,0.0,0.0,False
4,PRJ-001-ACT-005,PRJ-001,WBS-01,MEP installation,HVAC,2025-02-20,2025-04-03,2025-02-20,2025-04-03,100.0,Complete,False,P6_EXPORT,0.0,0.0,False


In [33]:
# ---------------------------------------------------------
# DERIVED PROCUREMENT METRICS
# ---------------------------------------------------------

procurement_clean["delivery_delay_days"] = (
    procurement_clean["actual_delivery_date"]
    - procurement_clean["planned_delivery_date"]
).dt.days

procurement_clean["delayed_delivery_flag"] = (
    procurement_clean["delivery_delay_days"] > 0
)

procurement_clean["critical_delay_flag"] = (
    (procurement_clean["delivery_delay_days"] > 14)
    & (procurement_clean["criticality"] == "High")
)

procurement_clean.head()

,procurement_id,project_code,package_code,vendor_id,category,rfq_date,po_date,planned_delivery_date,actual_delivery_date,package_value,status,criticality,source_system,delivery_delay_days,delayed_delivery_flag,critical_delay_flag
0,PO-00001,PRJ-001,PRJ-001-PKG-01,V-004,Civil,2025-08-22,2025-09-09,2025-12-29,2026-01-01,2405511.22,Delivered,Medium,PROCURE_TRACKER,3,True,False
1,PO-00002,PRJ-001,PRJ-001-PKG-02,V-003,Structural,2025-03-28,2025-04-11,2025-05-24,2025-05-29,1031731.65,Delivered,Medium,PROCURE_TRACKER,5,True,False
2,PO-00003,PRJ-001,PRJ-001-PKG-03,V-016,Mechanical,2025-07-28,2025-08-29,2025-10-24,2025-10-25,1945726.28,Delivered,Medium,PROCURE_TRACKER,1,True,False
3,PO-00004,PRJ-001,PRJ-001-PKG-04,V-023,Electrical,2025-03-22,2025-04-22,2025-08-10,2025-08-10,1716307.99,Delivered,Medium,PROCURE_TRACKER,0,False,False
5,PO-00006,PRJ-001,PRJ-001-PKG-05,V-024,HVAC,2025-04-25,2025-05-11,2025-09-04,2025-09-12,1305637.10,Delivered,Medium,PROCURE_TRACKER,8,True,False


In [34]:
# ---------------------------------------------------------
# CORRECT FORECAST VARIANCE
# ---------------------------------------------------------

costs_clean = costs_clean.merge(
    packages_clean[
        [
            "package_code",
            "package_budget"
        ]
    ],
    on="package_code",
    how="left"
)

costs_clean["forecast_variance"] = (
    costs_clean["package_budget"]
    - costs_clean["forecast_at_completion"]
)

costs_clean["forecast_variance_pct"] = np.where(
    costs_clean["package_budget"] != 0,
    (
        costs_clean["forecast_variance"]
        / costs_clean["package_budget"]
    ) * 100,
    np.nan
)

costs_clean.head()

,cost_record_id,project_code,package_code,cost_category,report_month,planned_cost,actual_cost,committed_cost,forecast_at_completion,currency,source_system,cost_variance,cost_variance_pct,forecast_variance,cost_anomaly_flag,package_budget,forecast_variance_pct
0,C-000001,PRJ-001,PRJ-001-PKG-01,Civil,2025-01-31,200572.03,191419.61,206308.06,6745303.66,GBP,ERP_COST,9152.42,4.563159,-252271.09,False,6493032.57,-3.885258
1,C-000002,PRJ-001,PRJ-001-PKG-02,Structural,2025-01-31,85691.16,85236.76,96286.32,2788355.37,GBP,ERP_COST,454.40,0.530276,-14312.18,False,2774043.19,-0.515932
2,C-000003,PRJ-001,PRJ-001-PKG-03,Mechanical,2025-01-31,161080.60,167524.71,167524.71,5433249.95,GBP,ERP_COST,-6444.11,-4.000550,-218656.44,False,5214593.51,-4.193164
3,C-000004,PRJ-001,PRJ-001-PKG-04,Electrical,2025-01-31,129328.27,126722.65,147595.69,4204065.36,GBP,ERP_COST,2605.62,2.014734,-17376.63,False,4186688.73,-0.415045
4,C-000005,PRJ-001,PRJ-001-PKG-05,HVAC,2025-01-31,52316.67,47323.64,60821.26,1775476.36,GBP,ERP_COST,4993.03,9.543860,-81851.22,False,1693625.14,-4.832901


In [35]:
# ---------------------------------------------------------
# VALIDATE CLEAN DATA
# ---------------------------------------------------------

print("COST DATA")
print("Rows:", len(costs_clean))
print("Missing project codes:", costs_clean["project_code"].isna().sum())
print("Negative actual costs:", (costs_clean["actual_cost"] < 0).sum())
print("Invalid project references:",
      (~costs_clean["project_code"].isin(valid_project_codes)).sum())


print("\nPROGRESS DATA")
print("Rows:", len(progress_clean))
print(
    "Invalid actual progress values:",
    (
        (progress_clean["actual_progress_pct"] < 0)
        |
        (progress_clean["actual_progress_pct"] > 100)
    ).sum()
)


print("\nSCHEDULE DATA")
print("Rows:", len(schedule_clean))
print(
    "Missing percent complete:",
    schedule_clean["percent_complete"].isna().sum()
)


print("\nPROCUREMENT DATA")
print("Rows:", len(procurement_clean))
print(
    "Missing vendor IDs:",
    procurement_clean["vendor_id"].isna().sum()
)

print(
    "Duplicate procurement IDs:",
    procurement_clean["procurement_id"].duplicated().sum()
)

COST DATA
Rows: 1821
Missing project codes: 0
Negative actual costs: 0
Invalid project references: 0

PROGRESS DATA
Rows: 151
Invalid actual progress values: 0

SCHEDULE DATA
Rows: 318
Missing percent complete: 0

PROCUREMENT DATA
Rows: 115
Missing vendor IDs: 0
Duplicate procurement IDs: 0


In [36]:
cost_anomalies = costs_clean[
    costs_clean["cost_anomaly_flag"] == True
]

cost_anomalies[
    [
        "cost_record_id",
        "project_code",
        "package_code",
        "planned_cost",
        "actual_cost",
        "cost_anomaly_flag",
    ]
]

,cost_record_id,project_code,package_code,planned_cost,actual_cost,cost_anomaly_flag
118,C-000121,PRJ-001,PRJ-001-PKG-01,363612.91,2732739.52,True


In [37]:
# ---------------------------------------------------------
# SAVE CLEAN PROCESSED DATA
# ---------------------------------------------------------

projects_clean.to_csv(
    PROCESSED / "projects_clean.csv",
    index=False
)

vendors_clean.to_csv(
    PROCESSED / "vendors_clean.csv",
    index=False
)

packages_clean.to_csv(
    PROCESSED / "cost_packages_clean.csv",
    index=False
)

costs_clean.to_csv(
    PROCESSED / "cost_monthly_clean.csv",
    index=False
)

progress_clean.to_csv(
    PROCESSED / "progress_updates_clean.csv",
    index=False
)

schedule_clean.to_csv(
    PROCESSED / "schedule_activities_clean.csv",
    index=False
)

procurement_clean.to_csv(
    PROCESSED / "procurement_packages_clean.csv",
    index=False
)

print("Clean processed datasets saved.")

Clean processed datasets saved.


In [38]:
# ---------------------------------------------------------
# SAVE QUARANTINED RECORDS
# ---------------------------------------------------------

costs_quarantine.to_csv(
    PROCESSED / "cost_monthly_quarantine.csv",
    index=False
)

progress_quarantine.to_csv(
    PROCESSED / "progress_updates_quarantine.csv",
    index=False
)

schedule_quarantine.to_csv(
    PROCESSED / "schedule_activities_quarantine.csv",
    index=False
)

procurement_quarantine.to_csv(
    PROCESSED / "procurement_packages_quarantine.csv",
    index=False
)

print("Quarantined records saved.")

Quarantined records saved.


In [39]:
# ---------------------------------------------------------
# SAVE DATA QUALITY AUDIT OUTPUTS
# ---------------------------------------------------------

quality_issues_df.to_csv(
    PROCESSED / "data_quality_issues.csv",
    index=False
)

quality_summary.to_csv(
    PROCESSED / "data_quality_summary.csv",
    index=False
)

print("Data quality audit files saved.")

Data quality audit files saved.


In [40]:
for file in sorted(PROCESSED.glob("*.csv")):
    df = pd.read_csv(file)

    print(
        f"{file.name:40s}"
        f"{len(df):5d} rows x "
        f"{len(df.columns):2d} columns"
    )

cost_monthly_clean.csv                   1821 rows x 17 columns
cost_monthly_quarantine.csv                 3 rows x 11 columns
cost_packages_clean.csv                    96 rows x  5 columns
data_quality_issues.csv                    13 rows x  6 columns
data_quality_summary.csv                   10 rows x  3 columns
procurement_packages_clean.csv            115 rows x 16 columns
procurement_packages_quarantine.csv         3 rows x 13 columns
progress_updates_clean.csv                151 rows x  8 columns
progress_updates_quarantine.csv             1 rows x  6 columns
projects_clean.csv                          8 rows x 10 columns
schedule_activities_clean.csv             318 rows x 16 columns
schedule_activities_quarantine.csv          2 rows x 13 columns
vendors_clean.csv                          30 rows x  4 columns


In [41]:
# ---------------------------------------------------------
# CREATE ETL AUDIT REPORT
# ---------------------------------------------------------

etl_audit = pd.DataFrame([
    {
        "dataset": "cost_monthly",
        "input_rows": len(costs_raw),
        "quarantined_rows": len(costs_quarantine),
        "duplicates_removed": 0,
        "output_rows": len(costs_clean),
    },
    {
        "dataset": "progress_updates",
        "input_rows": len(progress_raw),
        "quarantined_rows": len(progress_quarantine),
        "duplicates_removed": 0,
        "output_rows": len(progress_clean),
    },
    {
        "dataset": "schedule_activities",
        "input_rows": len(schedule_raw),
        "quarantined_rows": len(schedule_quarantine),
        "duplicates_removed": 0,
        "output_rows": len(schedule_clean),
    },
    {
        "dataset": "procurement_packages",
        "input_rows": len(procurement_raw),
        "quarantined_rows": len(procurement_quarantine),
        "duplicates_removed": duplicates_removed,
        "output_rows": len(procurement_clean),
    },
])

etl_audit

,dataset,input_rows,quarantined_rows,duplicates_removed,output_rows
0,cost_monthly,1824,3,0,1821
1,progress_updates,152,1,0,151
2,schedule_activities,320,2,0,318
3,procurement_packages,119,3,1,115


In [42]:
etl_audit["rejection_rate_pct"] = (
    etl_audit["quarantined_rows"]
    / etl_audit["input_rows"]
    * 100
).round(2)

etl_audit

,dataset,input_rows,quarantined_rows,duplicates_removed,output_rows,rejection_rate_pct
0,cost_monthly,1824,3,0,1821,0.16
1,progress_updates,152,1,0,151,0.66
2,schedule_activities,320,2,0,318,0.62
3,procurement_packages,119,3,1,115,2.52


In [43]:
etl_audit.to_csv(
    PROCESSED / "etl_audit_report.csv",
    index=False
)

print("ETL audit report saved.")

ETL audit report saved.


In [44]:
# ---------------------------------------------------------
# FINAL ETL RECONCILIATION CHECK
# ---------------------------------------------------------

etl_audit["reconciled"] = (
    etl_audit["input_rows"]
    ==
    (
        etl_audit["output_rows"]
        + etl_audit["quarantined_rows"]
        + etl_audit["duplicates_removed"]
    )
)

etl_audit

,dataset,input_rows,quarantined_rows,duplicates_removed,output_rows,rejection_rate_pct,reconciled
0,cost_monthly,1824,3,0,1821,0.16,True
1,progress_updates,152,1,0,151,0.66,True
2,schedule_activities,320,2,0,318,0.62,True
3,procurement_packages,119,3,1,115,2.52,True


In [45]:
print(
    "All ETL datasets reconciled:",
    etl_audit["reconciled"].all()
)

All ETL datasets reconciled: True


In [46]:
etl_audit.to_csv(
    PROCESSED / "etl_audit_report.csv",
    index=False
)

print("Final ETL audit report saved.")

Final ETL audit report saved.


In [47]:
for file in sorted(PROCESSED.glob("*.csv")):

    df = pd.read_csv(file)

    print(
        f"{file.name:40s}"
        f"{len(df):5d} rows x "
        f"{len(df.columns):2d} columns"
    )

cost_monthly_clean.csv                   1821 rows x 17 columns
cost_monthly_quarantine.csv                 3 rows x 11 columns
cost_packages_clean.csv                    96 rows x  5 columns
data_quality_issues.csv                    13 rows x  6 columns
data_quality_summary.csv                   10 rows x  3 columns
etl_audit_report.csv                        4 rows x  7 columns
procurement_packages_clean.csv            115 rows x 16 columns
procurement_packages_quarantine.csv         3 rows x 13 columns
progress_updates_clean.csv                151 rows x  8 columns
progress_updates_quarantine.csv             1 rows x  6 columns
projects_clean.csv                          8 rows x 10 columns
schedule_activities_clean.csv             318 rows x 16 columns
schedule_activities_quarantine.csv          2 rows x 13 columns
vendors_clean.csv                          30 rows x  4 columns
